# A1: Forecast Model Benchmark — LSTM vs ARIMA(2,1,2) vs Vanilla Transformer
(Reviewer #2, Comment 3)

Compares three forecasting models on the same held-out test set (time-based split: test ≥ 2023-12-18 00:00)  
to justify the LSTM choice. Reports MAE, RMSE, MAPE for both LMP price and solar generation.

| Model | Type | Key Params |
|---|---|---|
| **ARIMA(2,1,2)** | Statistical baseline | AR=2, I=1, MA=2, no seasonal term |
| **Vanilla Transformer** | Deep learning baseline | d\_model=32, heads=4, layers=2, ffn=64 |
| **LSTM (ours)** | Encoder-Decoder LSTM | hidden=32, layers=1, dropout=0.2 |

All deep learning models: Adam lr=1e-3, batch=32, max epochs=100, early stopping patience=10.

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
warnings.filterwarnings('ignore')

DATA_DIR     = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'raw'))
CODE_DIR     = os.path.abspath(os.path.join(os.getcwd(), '..', 'code', 'Forecast'))
RESULTS_BASE = os.path.abspath(os.path.join(os.getcwd(), '..', 'results', 'forecast'))

sys.path.insert(0, CODE_DIR)
from dataset import PriceForecastDataset
from train   import train_model
from model   import EncoderDecoderLSTM, VanillaTransformer

DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN         = 24
PRED_LEN        = 8
SAMPLES_PER_DAY = 96
WARMUP          = 3 * SAMPLES_PER_DAY   # 288
CONTEXT_LEN     = 100                   # ARIMA rolling context
SARIMA_CONTEXT  = 288                   # SARIMA init context (>=2 seasonal periods)
print(f'Device: {DEVICE}')

In [2]:
# ── Load data ─────────────────────────────────────────────────────────────
price_df = pd.read_csv(os.path.join(DATA_DIR, 'FinalData_FMM_LMP.csv'),
                       parse_dates=['INTERVALSTARTTIME_GMT'])
solar_df = pd.read_csv(os.path.join(DATA_DIR, 'Output_Solar_NetLoad_15min.csv'),
                       parse_dates=['INTERVALSTARTTIME_GMT'])

price_raw   = price_df['Price'].values.astype(float)
solar_raw   = solar_df['Solar'].values.astype(float)
price_times = price_df['INTERVALSTARTTIME_GMT'].values

min_len     = min(len(price_raw), len(solar_raw))
price_raw   = price_raw[:min_len]
solar_raw   = solar_raw[:min_len]
price_times = price_times[:min_len]

# ── Time-based split (same as original LSTM: test ≥ 2023-12-18 00:00) ────
TEST_START    = np.datetime64('2023-12-18T00:00:00')
test_raw_idx  = int(np.searchsorted(price_times, TEST_START))
test_ds_start = test_raw_idx - WARMUP        # first test dataset index

N_ds      = min_len - PRED_LEN - WARMUP + 1
train_end = int(0.8 * test_ds_start)         # 80% pre-test → train
val_end   = test_ds_start                    # val = [train_end, test_ds_start)

# ── Data verification ────────────────────────────────────────────────────
print('=== Data Verification ===')
print(f'Price rows : {len(price_raw)}  range: {price_raw.min():.2f} ~ {price_raw.max():.2f} $/MWh')
print(f'Solar rows : {len(solar_raw)}  range: {solar_raw.min():.0f} ~ {solar_raw.max():.0f} MW')
print(f'Common rows: {min_len}  ({min_len/SAMPLES_PER_DAY:.1f} days)')
print(f'Period     : {price_times[0]} → {price_times[-1]}')
print()
print(f'Test start : raw[{test_raw_idx}] = {price_times[test_raw_idx]}')
print(f'Dataset    : N={N_ds}  train={train_end}  val={val_end-train_end}  test={N_ds-val_end}')
print()
# Spot-check: verify a few raw values at key boundaries
print('Spot-check raw prices at split boundaries:')
print(f'  train_end  raw[{train_end+WARMUP}]: {price_raw[train_end+WARMUP]:.4f} $/MWh  ({price_times[train_end+WARMUP]})')
print(f'  test start raw[{test_raw_idx}]     : {price_raw[test_raw_idx]:.4f} $/MWh  ({price_times[test_raw_idx]})')
print(f'  data end   raw[{min_len-1}]        : {price_raw[-1]:.4f} $/MWh  ({price_times[-1]})')

=== Data Verification ===
Price rows : 38016  range: -81.96 ~ 2000.00 $/MWh
Solar rows : 38016  range: 0 ~ 19341 MW
Common rows: 38016  (396.0 days)
Period     : 2023-06-01T16:00:00.000000 → 2024-07-01T15:45:00.000000

Test start : raw[19136] = 2023-12-18T00:00:00.000000
Dataset    : N=37721  train=15078  val=3770  test=18873

Spot-check raw prices at split boundaries:
  train_end  raw[15366]: 38.1703 $/MWh  (2023-11-08T17:30:00.000000)
  test start raw[19136]     : 56.1609 $/MWh  (2023-12-18T00:00:00.000000)
  data end   raw[38015]        : 35.1385 $/MWh  (2024-07-01T15:45:00.000000)


In [3]:
# ── Metrics helpers ────────────────────────────────────────────────────────
def compute_metrics(targets, outputs):
    """targets, outputs: (N, 8) arrays in original units."""
    t, o = np.array(targets, dtype=float), np.array(outputs, dtype=float)
    mae  = float(np.mean(np.abs(t - o)))
    rmse = float(np.sqrt(np.mean((t - o) ** 2)))
    mask = np.abs(t) > 1e-3
    mape = float(np.mean(np.abs((t[mask] - o[mask]) / t[mask])) * 100) if mask.sum() > 0 else float('nan')
    return {'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mape}


def postprocess_solar(targets, preds):
    """
    Apply identical physical constraints to all three models before metrics:
      1. Clip predictions to >= 0  (solar generation cannot be negative)
      2. Force pred = 0 where target = 0  (night / no-sun periods excluded from MAPE)
    Applied to LSTM, ARIMA, and Transformer equally → fair comparison.
    """
    t = np.maximum(np.array(targets, dtype=float), 0.0)
    p = np.maximum(np.array(preds,   dtype=float), 0.0)
    p[t == 0.0] = 0.0
    return t, p

## 1. Deep Learning Models: LSTM & Vanilla Transformer

Both models share **identical** batch structure:

| Item | Value |
|---|---|
| Encoder input | `(B, 24, 1)` — 6-hour price/solar history |
| Decoder input | `(B, 8, 3)` — same-slot prices from previous 3 days |
| Output | `(B, 8, 1)` — 8-step (2-hour) forecast, **all slots predicted simultaneously** |
| Loss | `MSELoss` over all 8 output slots at once |
| Optimizer | Adam, lr=1e-3 |
| Batch size | 32 (train/val), 64 (test) |
| Max epochs | 100, early stopping patience=10 |

ARIMA processes one time series sample at a time (no batch concept).

In [4]:
def train_or_load_model(raw_data, tag, model_class,
                        n_ds, train_end, val_end,
                        results_base=RESULTS_BASE, device=DEVICE):
    """
    Unified train/load for EncoderDecoderLSTM and VanillaTransformer.
    Both models use identical DataLoaders and MSELoss over all 8 output slots.

    tag         : e.g. 'LSTM_Price', 'Transformer_Solar'
    model_class : EncoderDecoderLSTM | VanillaTransformer
    Returns (targets_orig, preds_orig): (N_test, 8) in raw units.
    """
    pred_csv  = os.path.join(results_base, f'predictions_{tag}.csv')
    ckpt_path = os.path.join(results_base, f'best_{tag}.pth')

    if os.path.exists(pred_csv):
        print(f'[{tag}] Loading saved predictions.')
        df = pd.read_csv(pred_csv)
        return (df[[f'target_{i+1}' for i in range(PRED_LEN)]].values,
                df[[f'output_{i+1}' for i in range(PRED_LEN)]].values)

    ds = PriceForecastDataset(raw_data, np.arange(len(raw_data)))

    train_dl = DataLoader(Subset(ds, range(train_end)),           batch_size=32, shuffle=True)
    val_dl   = DataLoader(Subset(ds, range(train_end, val_end)),  batch_size=32)
    test_dl  = DataLoader(Subset(ds, range(val_end,   n_ds)),     batch_size=64)

    print(f'[{tag}] Training {model_class.__name__}  '
          f'(train={train_end}  val={val_end-train_end}  test={n_ds-val_end})')

    model = model_class().to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit  = nn.MSELoss()
    model, best_val = train_model(model, train_dl, val_dl, crit, opt, device)
    torch.save(model.state_dict(), ckpt_path)
    print(f'[{tag}] best_val_mse={best_val:.6f}')

    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for enc, dec, tgt_batch in test_dl:
            enc, dec = enc.to(device), dec.to(device)
            pred = model(enc, dec).squeeze(-1).cpu().numpy()
            y    = tgt_batch['prices'].squeeze(-1).cpu().numpy()
            all_preds.append(pred)
            all_targets.append(y)

    preds_s   = np.vstack(all_preds)
    targets_s = np.vstack(all_targets)

    preds_orig   = ds.inverse_transform(preds_s.flatten()).reshape(-1, PRED_LEN)
    targets_orig = ds.inverse_transform(targets_s.flatten()).reshape(-1, PRED_LEN)

    raw_check = raw_data[val_end + WARMUP]
    csv_check = targets_orig[0, 0]
    print(f'[{tag}] Alignment check: raw[{val_end+WARMUP}]={raw_check:.4f}  '
          f'csv_target_1={csv_check:.4f}  ratio={csv_check/raw_check:.4f}')

    df_out = pd.DataFrame({
        **{f'target_{k+1}': targets_orig[:, k] for k in range(PRED_LEN)},
        **{f'output_{k+1}': preds_orig[:, k]   for k in range(PRED_LEN)},
    })
    df_out.to_csv(pred_csv, index=False)
    print(f'[{tag}] Saved → {pred_csv}')
    return targets_orig, preds_orig

# ── Train / load LSTM ─────────────────────────────────────────────────────
lstm_price_tgt, lstm_price_pred = train_or_load_model(
    price_raw, 'LSTM_Price', EncoderDecoderLSTM, N_ds, train_end, val_end)
lstm_price_m = compute_metrics(lstm_price_tgt, lstm_price_pred)
print('LSTM Price:', lstm_price_m)

lstm_solar_tgt, lstm_solar_pred = train_or_load_model(
    solar_raw, 'LSTM_Solar', EncoderDecoderLSTM, N_ds, train_end, val_end)
_t, _p = postprocess_solar(lstm_solar_tgt, lstm_solar_pred)
lstm_solar_m = compute_metrics(_t, _p)
print('LSTM Solar (post-processed):', lstm_solar_m)

[LSTM_Price] Training EncoderDecoderLSTM  (train=15078  val=3770  test=18873)
Epoch   1  train=0.603599  val=0.015077  best=0.015077  *
Epoch   2  train=0.384248  val=0.009712  best=0.009712  *
Epoch   3  train=0.349194  val=0.015728  best=0.009712
Epoch   4  train=0.302227  val=0.007214  best=0.007214  *
Epoch   5  train=0.281882  val=0.006416  best=0.006416  *
Epoch   6  train=0.269190  val=0.006507  best=0.006416
Epoch   7  train=0.266013  val=0.005827  best=0.005827  *
Epoch   8  train=0.268572  val=0.006759  best=0.005827
Epoch   9  train=0.245133  val=0.009982  best=0.005827
Epoch  10  train=0.249522  val=0.006798  best=0.005827
Epoch  11  train=0.239799  val=0.006680  best=0.005827
Epoch  12  train=0.236748  val=0.008014  best=0.005827
Epoch  13  train=0.227361  val=0.006866  best=0.005827
Epoch  14  train=0.228470  val=0.005852  best=0.005827
Epoch  15  train=0.226173  val=0.005474  best=0.005474  *
Epoch  16  train=0.227841  val=0.048630  best=0.005474
Epoch  17  train=0.21625

## 2. Statistical Baselines: ARIMA(2,1,2) & SARIMA(2,1,2)(0,0,1)[96]

**ARIMA(2,1,2)**: non-seasonal statistical baseline. Fits once on training data;  
rolling 100-step context window at test time.

**SARIMA(2,1,2)(0,0,1)[96]**: adds a seasonal MA term at lag-96 (daily cycle at 15-min resolution).  
No seasonal differencing (D=0) — computationally feasible for large s=96.  
The lag-96 MA coefficient captures the residual daily seasonal correlation after ARIMA(2,1,2) filtering.  
Rolling Kalman-state update: state initialised on 288 obs, advanced one true observation at a time.

In [5]:
import time

# ── ARIMA helpers ──────────────────────────────────────────────────────────
def fit_arima(data, order=(2, 1, 2)):
    res = ARIMA(data, order=order, trend='n').fit(method_kwargs={'warn_convergence': False})
    p, d, q = order
    params = np.array(res.params)
    ar = params[:p];  ma = params[p:p + q]
    print(f'ARIMA{order}  AR={ar.round(4)}  MA={ma.round(4)}')
    return ar, ma


def arima_forecast(history, ar, ma, h=8):
    d = np.diff(history);  n = len(d);  p, q = len(ar), len(ma)
    res = np.zeros(n)
    for t in range(max(p, q), n):
        fitted = sum(ar[i] * d[t-1-i] for i in range(p)) \
               + sum(ma[j] * res[t-1-j] for j in range(q))
        res[t] = d[t] - fitted
    ext_d, ext_res = list(d), list(res)
    for step in range(h):
        ar_term = sum(ar[i] * ext_d[-1-i] for i in range(p))
        ma_term = sum(ma[j] * ext_res[n - 1 - (j - step)] for j in range(step, q))
        ext_d.append(ar_term + ma_term);  ext_res.append(0.0)
    return history[-1] + np.cumsum(np.array(ext_d[n:]))


def run_arima_test(raw_data, ar, ma, n_ds, val_end_ds,
                   seq_len=SEQ_LEN, pred_len=PRED_LEN):
    warmup = 3 * SAMPLES_PER_DAY
    preds, targets = [], []
    for i, k in enumerate(range(val_end_ds, n_ds)):
        t_start = k + warmup
        ctx_s   = max(0, t_start - CONTEXT_LEN)
        preds.append(arima_forecast(raw_data[ctx_s:t_start], ar, ma, h=pred_len))
        targets.append(raw_data[t_start:t_start + pred_len])
        if (i + 1) % 5000 == 0:
            print(f'  {i+1}/{n_ds - val_end_ds} done')
    return np.array(targets), np.array(preds)


# ── SARIMA helpers ──────────────────────────────────────────────────────────
SARIMA_ORDER    = (2, 1, 2)
SARIMA_SEASONAL = (0, 0, 1, 96)   # seasonal MA at lag-96, no seasonal differencing
SARIMA_MAXITER  = 100


def _sarima_callback(maxiter):
    state = {'n': 0, 't0': time.time()}
    def cb(xk):
        state['n'] += 1
        elapsed = time.time() - state['t0']
        print(f'\r  iter {state["n"]:3d}/{maxiter}  elapsed {elapsed:.0f}s', end='', flush=True)
    return cb


def fit_sarima(data, order=SARIMA_ORDER, seasonal_order=SARIMA_SEASONAL):
    print(f'Fitting SARIMA{order}x{seasonal_order} on {len(data)} obs ...')
    cb = _sarima_callback(SARIMA_MAXITER)
    res = SARIMAX(data, order=order, seasonal_order=seasonal_order, trend='n').fit(
        disp=False, maxiter=SARIMA_MAXITER,
        method_kwargs={'warn_convergence': False},
        callback=cb)
    print(f'\n  AIC={res.aic:.2f}  BIC={res.bic:.2f}')
    return res


def run_sarima_test(raw_data, sarima_res, n_ds, val_end_ds,
                    pred_len=PRED_LEN, init_len=SARIMA_CONTEXT):
    """
    Fast vectorized SARIMA rolling forecast.

    Runs the Kalman FILTER (not smoother) once on context + test data.
    apply() internally calls smooth() which allocates (state_dim, state_dim, n_obs)
    — 1.4 GB OOM. Instead we call filter() with filter_output=1 (FILTER_STATE only),
    storing only the filtered state vector (~15 MB) and nothing else.
    h-step conditional forecast: E[y_{t+h}|y_1..y_t] = Z @ T^h @ filtered_state_t
    """
    warmup  = 3 * SAMPLES_PER_DAY
    t_start = val_end_ds + warmup
    n_test  = n_ds - val_end_ds

    ctx_start = max(0, t_start - init_len)
    full_data = raw_data[ctx_start : t_start + n_test]
    offset    = t_start - ctx_start

    print(f'  Kalman filter on {len(full_data)} obs (FILTER_STATE only, no smoother) ...')
    t0 = time.time()

    new_mod = SARIMAX(full_data,
                      order=sarima_res.model.order,
                      seasonal_order=sarima_res.model.seasonal_order,
                      trend='n')
    new_mod.ssm.filter_output = 1   # FILTER_STATE = 0x01 — skip covariance storage
    full_filt = new_mod.filter(sarima_res.params)
    print(f'  done ({time.time()-t0:.1f}s)  building forecasts ...')

    fr = full_filt.filter_results
    T  = fr.transition[:, :, 0]
    Z  = fr.design[0, :, 0]
    d  = float(fr.obs_intercept[0, 0])

    # Pre-compute T^1 ... T^pred_len
    T_powers, Tpow = [], np.eye(T.shape[0])
    for _ in range(pred_len):
        Tpow = Tpow @ T
        T_powers.append(Tpow)

    # State available before predicting test step i = filtered_state[:, offset+i-1]
    S = fr.filtered_state[:, offset - 1 : offset - 1 + n_test]   # (m, n_test)

    preds = np.zeros((n_test, pred_len))
    for h in range(pred_len):
        preds[:, h] = Z @ T_powers[h] @ S + d

    targets = np.stack([raw_data[t_start + i : t_start + i + pred_len]
                        for i in range(n_test)])
    print(f'  done ({time.time()-t0:.1f}s total)')
    return targets, preds


def sarima_with_cache(raw_data, tag, sarima_res, n_ds, val_end_ds,
                      results_base=RESULTS_BASE):
    pred_csv = os.path.join(results_base, f'predictions_{tag}.csv')
    if os.path.exists(pred_csv):
        print(f'[{tag}] Loading saved predictions.')
        df = pd.read_csv(pred_csv)
        return (df[[f'target_{i+1}' for i in range(PRED_LEN)]].values,
                df[[f'output_{i+1}' for i in range(PRED_LEN)]].values)
    tgt, pred = run_sarima_test(raw_data, sarima_res, n_ds, val_end_ds)
    pd.DataFrame({
        **{f'target_{k+1}': tgt[:, k] for k in range(PRED_LEN)},
        **{f'output_{k+1}': pred[:, k] for k in range(PRED_LEN)},
    }).to_csv(pred_csv, index=False)
    print(f'[{tag}] Saved -> {pred_csv}')
    return tgt, pred

In [6]:
raw_train_len = train_end + WARMUP

print('=== Price ARIMA ===')
price_ar, price_ma = fit_arima(price_raw[:raw_train_len])
print('\n=== Solar ARIMA ===')
solar_ar, solar_ma = fit_arima(solar_raw[:raw_train_len])

print('\n=== Price SARIMA ===')
price_sarima_res = fit_sarima(price_raw[:raw_train_len])

print('\n=== Solar SARIMA ===')
_solar_sarima_cache = os.path.join(RESULTS_BASE, 'predictions_SARIMA_Solar.csv')
if os.path.exists(_solar_sarima_cache):
    print('[SARIMA_Solar] Cache exists - skipping fit to avoid OOM.')
    solar_sarima_res = None
else:
    solar_sarima_res = fit_sarima(solar_raw[:raw_train_len])

=== Price ARIMA ===
ARIMA(2, 1, 2)  AR=[ 0.1616 -0.7747]  MA=[-0.2762  0.708 ]

=== Solar ARIMA ===
ARIMA(2, 1, 2)  AR=[ 1.5377 -0.6172]  MA=[-0.4909  0.1649]

=== Price SARIMA ===
Fitting SARIMA(2, 1, 2)x(0, 0, 1, 96) on 15366 obs ...
  iter  31/100  elapsed 78s
  AIC=151255.92  BIC=151301.76

=== Solar SARIMA ===
Fitting SARIMA(2, 1, 2)x(0, 0, 1, 96) on 15366 obs ...
  iter  29/100  elapsed 76s
  AIC=207237.06  BIC=207282.90


In [7]:
print('Generating ARIMA price forecasts ...')
arima_price_tgt, arima_price_pred = run_arima_test(
    price_raw, price_ar, price_ma, N_ds, val_end_ds=test_ds_start)
arima_price_m = compute_metrics(arima_price_tgt, arima_price_pred)
print('ARIMA Price:', arima_price_m)

print('\nGenerating ARIMA solar forecasts ...')
arima_solar_tgt, arima_solar_pred = run_arima_test(
    solar_raw, solar_ar, solar_ma, N_ds, val_end_ds=test_ds_start)
_t, _p = postprocess_solar(arima_solar_tgt, arima_solar_pred)
arima_solar_m = compute_metrics(_t, _p)
print('ARIMA Solar (post-processed):', arima_solar_m)

print('\nGenerating SARIMA price forecasts ...')
sarima_price_tgt, sarima_price_pred = sarima_with_cache(
    price_raw, 'SARIMA_Price', price_sarima_res, N_ds, test_ds_start)
sarima_price_m = compute_metrics(sarima_price_tgt, sarima_price_pred)
print('SARIMA Price:', sarima_price_m)

print('\nGenerating SARIMA solar forecasts ...')
if solar_sarima_res is None:
    _df_ss = pd.read_csv(_solar_sarima_cache)
    sarima_solar_tgt  = _df_ss[[f'target_{i+1}' for i in range(PRED_LEN)]].values
    sarima_solar_pred = _df_ss[[f'output_{i+1}' for i in range(PRED_LEN)]].values
    print('[SARIMA_Solar] Loaded from cache.')
else:
    sarima_solar_tgt, sarima_solar_pred = sarima_with_cache(
        solar_raw, 'SARIMA_Solar', solar_sarima_res, N_ds, test_ds_start)
_t, _p = postprocess_solar(sarima_solar_tgt, sarima_solar_pred)
sarima_solar_m = compute_metrics(_t, _p)
print('SARIMA Solar (post-processed):', sarima_solar_m)

Generating ARIMA price forecasts ...
  5000/18873 done
  10000/18873 done
  15000/18873 done
ARIMA Price: {'MAE': 9.626628276427606, 'RMSE': 18.71714195927734, 'MAPE (%)': 574.6670767078118}

Generating ARIMA solar forecasts ...
  5000/18873 done
  10000/18873 done
  15000/18873 done
ARIMA Solar (post-processed): {'MAE': 945.3326462565941, 'RMSE': 2076.9521082926367, 'MAPE (%)': 340.153102805824}

Generating SARIMA price forecasts ...
  Kalman filter on 19161 obs (FILTER_STATE only, no smoother) ...
  done (40.3s)  building forecasts ...
  done (40.4s total)
[SARIMA_Price] Saved -> C:\Users\WJ\OneDrive\바탕 화면\Research_Real time energy bid\APEN_Major_Revision\results\predictions_SARIMA_Price.csv
SARIMA Price: {'MAE': 8.800840763394133, 'RMSE': 17.33751268941211, 'MAPE (%)': 497.6818451856111}

Generating SARIMA solar forecasts ...
  Kalman filter on 19161 obs (FILTER_STATE only, no smoother) ...
  done (40.8s)  building forecasts ...
  done (40.8s total)
[SARIMA_Solar] Saved -> C:\Users\

In [8]:
tf_price_tgt, tf_price_pred = train_or_load_model(
    price_raw, 'Transformer_Price', VanillaTransformer, N_ds, train_end, val_end)
tf_price_m = compute_metrics(tf_price_tgt, tf_price_pred)
print('Transformer Price:', tf_price_m)

tf_solar_tgt, tf_solar_pred = train_or_load_model(
    solar_raw, 'Transformer_Solar', VanillaTransformer, N_ds, train_end, val_end)
_t, _p = postprocess_solar(tf_solar_tgt, tf_solar_pred)
tf_solar_m = compute_metrics(_t, _p)
print('Transformer Solar (post-processed):', tf_solar_m)

[Transformer_Price] Training VanillaTransformer  (train=15078  val=3770  test=18873)
Epoch   1  train=5.699007  val=0.031214  best=0.031214  *
Epoch   2  train=0.882754  val=0.038506  best=0.031214
Epoch   3  train=0.671998  val=0.039363  best=0.031214
Epoch   4  train=0.610077  val=0.025043  best=0.025043  *
Epoch   5  train=0.569911  val=0.023987  best=0.023987  *
Epoch   6  train=0.523939  val=0.019572  best=0.019572  *
Epoch   7  train=0.466466  val=0.023248  best=0.019572
Epoch   8  train=0.465988  val=0.020750  best=0.019572
Epoch   9  train=0.436992  val=0.030718  best=0.019572
Epoch  10  train=0.438387  val=0.020704  best=0.019572
Epoch  11  train=0.422627  val=0.018626  best=0.018626  *
Epoch  12  train=0.433581  val=0.021884  best=0.018626
Epoch  13  train=0.421166  val=0.023069  best=0.018626
Epoch  14  train=0.423222  val=0.023538  best=0.018626
Epoch  15  train=0.406100  val=0.022479  best=0.018626
Epoch  16  train=0.384543  val=0.014685  best=0.014685  *
Epoch  17  train=

## 3. Comparison

In [9]:
def make_table(arima_m, sarima_m, tf_m, lstm_m, unit):
    return pd.DataFrame({
        'Model'          : ['ARIMA(2,1,2)', 'SARIMA(2,1,2)(0,0,1)[96]',
                            'Vanilla Transformer', 'LSTM (ours)'],
        f'MAE ({unit})'  : [arima_m['MAE'],      sarima_m['MAE'],
                            tf_m['MAE'],          lstm_m['MAE']],
        f'RMSE ({unit})' : [arima_m['RMSE'],     sarima_m['RMSE'],
                            tf_m['RMSE'],         lstm_m['RMSE']],
        'MAPE (%)'       : [arima_m['MAPE (%)'],  sarima_m['MAPE (%)'],
                            tf_m['MAPE (%)'],     lstm_m['MAPE (%)']],
    })

price_table = make_table(arima_price_m, sarima_price_m, tf_price_m, lstm_price_m, '$/MWh')
solar_table = make_table(arima_solar_m, sarima_solar_m, tf_solar_m, lstm_solar_m, 'MW')

print('=== LMP Price Forecast Comparison ===')
print(price_table.to_string(index=False, float_format='%.4f'))
print('\n=== Solar Generation Forecast Comparison ===')
print(solar_table.to_string(index=False, float_format='%.4f'))

price_table.to_csv(os.path.join(RESULTS_BASE, 'Forecast_Benchmark_Price.csv'), index=False)
solar_table.to_csv(os.path.join(RESULTS_BASE, 'Forecast_Benchmark_Solar.csv'), index=False)

=== LMP Price Forecast Comparison ===
                   Model  MAE ($/MWh)  RMSE ($/MWh)  MAPE (%)
            ARIMA(2,1,2)       9.6266       18.7171  574.6671
SARIMA(2,1,2)(0,0,1)[96]       8.8008       17.3375  497.6818
     Vanilla Transformer      19.0151       26.4705 1280.3747
             LSTM (ours)      10.4836       17.7607  681.0264

=== Solar Generation Forecast Comparison ===
                   Model  MAE (MW)  RMSE (MW)  MAPE (%)
            ARIMA(2,1,2)  945.3326  2076.9521  340.1531
SARIMA(2,1,2)(0,0,1)[96]  793.5545  1712.2358  261.4165
     Vanilla Transformer  551.3808  1020.2854  106.3400
             LSTM (ours)  363.3236   691.0926   84.1348


In [10]:
# ── Build timestamps (one per test sample = raw index of slot-1 target) ──────
timestamps = np.array([price_times[test_ds_start + WARMUP + i] for i in range(N_ds - test_ds_start)])

# ── Postprocess all solar predictions identically ────────────────────────────
_, ap_s = postprocess_solar(arima_solar_tgt,  arima_solar_pred)
_, sp_s = postprocess_solar(sarima_solar_tgt, sarima_solar_pred)
_, tp_s = postprocess_solar(tf_solar_tgt,     tf_solar_pred)
_, lp_s = postprocess_solar(lstm_solar_tgt,   lstm_solar_pred)
at_s    = np.maximum(lstm_solar_tgt.astype(float), 0.0)  # single actual reference


def build_compare_df(timestamps, actual, arima_p, sarima_p, tf_p, lstm_p):
    """One row per test sample. Columns: timestamp | actual_slot1..8 | {model}_slot1..8 x4"""
    rows = {'timestamp': timestamps}
    for s in range(PRED_LEN):
        rows[f'actual_slot{s+1}']      = actual[:, s]
        rows[f'arima_slot{s+1}']       = arima_p[:, s]
        rows[f'sarima_slot{s+1}']      = sarima_p[:, s]
        rows[f'transformer_slot{s+1}'] = tf_p[:, s]
        rows[f'lstm_slot{s+1}']        = lstm_p[:, s]
    return pd.DataFrame(rows)


price_compare = build_compare_df(
    timestamps,
    lstm_price_tgt,      # all models share the same actual
    arima_price_pred, sarima_price_pred, tf_price_pred, lstm_price_pred,
)

# Solar arrays may differ in length if any model used cached predictions from a different run.
# Truncate all to the minimum available length for consistent comparison.
N_solar = min(len(at_s), len(ap_s), len(sp_s), len(tp_s), len(lp_s))
solar_compare = build_compare_df(
    timestamps[:N_solar],
    at_s[:N_solar],
    ap_s[:N_solar], sp_s[:N_solar], tp_s[:N_solar], lp_s[:N_solar],
)

price_path = os.path.join(RESULTS_BASE, 'compare_price_4models.csv')
solar_path = os.path.join(RESULTS_BASE, 'compare_solar_4models.csv')
price_compare.to_csv(price_path, index=False)
solar_compare.to_csv(solar_path, index=False)

print(f'Saved: {price_path}')
print(f'       {len(price_compare)} rows  |  columns: timestamp + actual x8 + arima x8 + sarima x8 + transformer x8 + lstm x8')
print(f'Saved: {solar_path}')
print(f'       {len(solar_compare)} rows  (solar: postprocessed)')

# ── Quick peek: slot-1 only ───────────────────────────────────────────────────
slot1 = ['timestamp', 'actual_slot1', 'arima_slot1', 'sarima_slot1', 'transformer_slot1', 'lstm_slot1']
print('\n=== Price slot-1 (first 5 rows, $/MWh) ===')
print(price_compare[slot1].head().to_string(index=False, float_format='%.2f'))
print('\n=== Solar slot-1 (first 5 rows, MW) ===')
print(solar_compare[slot1].head().to_string(index=False, float_format='%.1f'))

Saved: C:\Users\WJ\OneDrive\바탕 화면\Research_Real time energy bid\APEN_Major_Revision\results\compare_price_4models.csv
       18873 rows  |  columns: timestamp + actual x8 + arima x8 + sarima x8 + transformer x8 + lstm x8
Saved: C:\Users\WJ\OneDrive\바탕 화면\Research_Real time energy bid\APEN_Major_Revision\results\compare_solar_4models.csv
       18873 rows  (solar: postprocessed)

=== Price slot-1 (first 5 rows, $/MWh) ===
          timestamp  actual_slot1  arima_slot1  sarima_slot1  transformer_slot1  lstm_slot1
2023-12-18 00:00:00         56.16        48.39         48.79              50.26       51.24
2023-12-18 00:15:00         44.88        55.55         55.38              49.50       52.97
2023-12-18 00:30:00         44.88        46.19         44.08              47.97       47.91
2023-12-18 00:45:00         32.56        46.43         44.09              46.95       44.34
2023-12-18 01:00:00         36.79        33.47         36.40              45.80       41.87

=== Solar slot-1 (firs

## 5. Architecture & Hyperparameter Summary (for paper)

In [11]:
arch_summary = pd.DataFrame([
    {'Parameter': 'Model type',
     'ARIMA(2,1,2)': 'Statistical (ARIMA)',
     'SARIMA(2,1,2)(0,0,1)[96]': 'Statistical (SARIMA)',
     'Vanilla Transformer': 'Encoder-Decoder Transformer',
     'LSTM (ours)': 'Encoder-Decoder LSTM'},
    {'Parameter': 'Seasonal component',
     'ARIMA(2,1,2)': 'None',
     'SARIMA(2,1,2)(0,0,1)[96]': 'Daily (s=96), D=0, Q=1',
     'Vanilla Transformer': 'Via decoder context',
     'LSTM (ours)': 'Via decoder context'},
    {'Parameter': 'Input (encoder)',
     'ARIMA(2,1,2)': '100-step rolling context',
     'SARIMA(2,1,2)(0,0,1)[96]': 'Running Kalman state',
     'Vanilla Transformer': '24-step history (6 h)',
     'LSTM (ours)': '24-step history (6 h)'},
    {'Parameter': 'Input (decoder)',
     'ARIMA(2,1,2)': '-',
     'SARIMA(2,1,2)(0,0,1)[96]': '-',
     'Vanilla Transformer': '3-day same-slot context',
     'LSTM (ours)': '3-day same-slot context'},
    {'Parameter': 'Output horizon',
     'ARIMA(2,1,2)': '8 steps (2 h)',
     'SARIMA(2,1,2)(0,0,1)[96]': '8 steps (2 h)',
     'Vanilla Transformer': '8 steps (2 h)',
     'LSTM (ours)': '8 steps (2 h)'},
    {'Parameter': 'Hidden / d_model',
     'ARIMA(2,1,2)': 'AR=2, MA=2',
     'SARIMA(2,1,2)(0,0,1)[96]': 'AR=2, MA=2, SMA=1',
     'Vanilla Transformer': '32',
     'LSTM (ours)': '32'},
    {'Parameter': 'Attention heads',
     'ARIMA(2,1,2)': '-',
     'SARIMA(2,1,2)(0,0,1)[96]': '-',
     'Vanilla Transformer': '4',
     'LSTM (ours)': '-'},
    {'Parameter': 'Layers',
     'ARIMA(2,1,2)': '-',
     'SARIMA(2,1,2)(0,0,1)[96]': '-',
     'Vanilla Transformer': '2 enc + 2 dec',
     'LSTM (ours)': '1'},
    {'Parameter': 'Dropout',
     'ARIMA(2,1,2)': '-',
     'SARIMA(2,1,2)(0,0,1)[96]': '-',
     'Vanilla Transformer': '0.1',
     'LSTM (ours)': '0.2'},
    {'Parameter': 'Optimizer',
     'ARIMA(2,1,2)': 'Max-likelihood (BFGS)',
     'SARIMA(2,1,2)(0,0,1)[96]': 'Max-likelihood (BFGS)',
     'Vanilla Transformer': 'Adam (lr=1e-3)',
     'LSTM (ours)': 'Adam (lr=1e-3)'},
    {'Parameter': 'Batch size',
     'ARIMA(2,1,2)': '-',
     'SARIMA(2,1,2)(0,0,1)[96]': '-',
     'Vanilla Transformer': '32',
     'LSTM (ours)': '32'},
    {'Parameter': 'Max epochs / patience',
     'ARIMA(2,1,2)': '-',
     'SARIMA(2,1,2)(0,0,1)[96]': '-',
     'Vanilla Transformer': '100 / 10',
     'LSTM (ours)': '100 / 10'},
])

print(arch_summary.to_string(index=False))
arch_summary.to_csv(os.path.join(RESULTS_BASE, 'Forecast_Architecture_Summary.csv'), index=False)

            Parameter             ARIMA(2,1,2) SARIMA(2,1,2)(0,0,1)[96]         Vanilla Transformer             LSTM (ours)
           Model type      Statistical (ARIMA)     Statistical (SARIMA) Encoder-Decoder Transformer    Encoder-Decoder LSTM
   Seasonal component                     None   Daily (s=96), D=0, Q=1         Via decoder context     Via decoder context
      Input (encoder) 100-step rolling context     Running Kalman state       24-step history (6 h)   24-step history (6 h)
      Input (decoder)                        -                        -     3-day same-slot context 3-day same-slot context
       Output horizon            8 steps (2 h)            8 steps (2 h)               8 steps (2 h)           8 steps (2 h)
     Hidden / d_model               AR=2, MA=2        AR=2, MA=2, SMA=1                          32                      32
      Attention heads                        -                        -                           4                       -
        

## 6. SAC / BC Input Files (LMP_Bid_LSTM, LMP_Ope_LSTM, Solar_Bid_LSTM)

Generate SAC input files from the new LSTM predictions and build the orig-vs-LSTM comparison CSV.

In [ ]:
import subprocess, sys

SCRIPTS_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'code', 'Forecast'))
PY = sys.executable  # use the same python kernel that is running this notebook

print('=== Generating SAC input files from LSTM predictions ===')
r = subprocess.run([PY, os.path.join(SCRIPTS_DIR, 'gen_sac_inputs_LSTM.py')],
                   capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('[ERROR]', r.stderr)

print()
print('=== Building orig vs LSTM comparison CSV ===')
r2 = subprocess.run([PY, os.path.join(SCRIPTS_DIR, 'make_comparison_csv.py')],
                    capture_output=True, text=True)
print(r2.stdout)
if r2.returncode != 0:
    print('[ERROR]', r2.stderr)